# Suite ALG — The predicate algebra

`Predicates` extends mbse-expressions' Basic with terms about a store's data: quantifiers over a schema's objects
(`forall`, `exists`, `count` over an `extent`), `linked`, and weighted choices (`choice` of `option`s). Mandatory,
possible and forbidden links are predicates of the algebra.

In [ ]:
from mbse.Expressions import Expressions as E
from mbse.Patterns import Constraints as C, Predicates as P, Queries as Q, Validators as V
from mbse.Expressions.Framework import Terms
from mbse.Schemas.Framework import JSON, Proxies, Schemas as S
from support import Contact, Phone, book, listed, raises

c, p, d, x = E.variable("c"), E.variable("p"), E.variable("d"), E.variable("x")
owns = P.exists("p", Phone, P.linked(c, "phones", p))
mandatory = P.forall("c", Contact, owns)  # every contact has a phone
forbidden = P.forall("c", Contact, owns.not_())  # no contact has one
possible = P.forall("c", Contact, P.choice((0.35, owns), (0.65, owns.not_())))  # 35% of contacts have one

## ALG-01 · The algebra's terms are written and checked

In [ ]:
assert isinstance(mandatory.data, P.OfForall) and mandatory.data.name == "c" and mandatory.data.collection.schema == "Contact"
assert P.extent("Contact").data.schema == P.extent(Contact).data.schema == "Contact"  # a schema, or its name
assert P.DIALECT.base is E.DIALECT and P.DIALECT.validate(mandatory.data, core=True) == []
assert list(P.DIALECT.kinds())[-7:] == ["extent", "forall", "exists", "count", "linked", "choice", "option"]
assert [n for n in P.DIALECT.schemas if n.startswith("Patterns.")] == [
    "Patterns.OfExtent", "Patterns.OfForall", "Patterns.OfExists", "Patterns.OfCount", "Patterns.OfLinked",
    "Patterns.OfChoice", "Patterns.OfOption"]
assert E.DIALECT.validate(mandatory.data)[0].startswith("not an expression: ")  # Basic alone cannot check it
mixed = c.has("name").and_(owns)  # Basic's writers hold the algebra's terms, and its writers Basic's
assert P.DIALECT.validate(mixed.data, bound={"c"}, core=True) == [] and isinstance(owns.not_().data, E.OfOperation.Data)
assert P.DIALECT.validate(P.linked(c, "phones", p).data) == ["source: variable 'c' is not bound",
                                                              "target: variable 'p' is not bound"]
assert P.DIALECT.validate(P.choice((0.5, True), (0.4, False)).data) == ["a choice's weights must sum to 1, got 0.9"]
assert P.DIALECT.validate(P.choice((-1.0, True), (2.0, False)).data) == [
    "argument 0: an option's weight must be positive, got -1.0"]
assert P.DIALECT.validate(P.choice().data) == ["a choice needs an option"]
assert P.DIALECT.validate(P.OfChoice((E.literal(True).data,))) == ["a choice's arguments are options"]
with raises(TypeError, match="expected a named schema or its name, got "):
    P.extent(S.OfObject.Builder().create())

## ALG-02 · The algebra is evaluated over the store

In [ ]:
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda q: q.number("1"))).create()
bob = store.Contact().name("bob").create()
listed(store, ann, bob)
evaluate = P.Evaluator(store)
assert evaluate(mandatory) is False and evaluate(forbidden) is False and evaluate(possible) is True  # bob has none
assert evaluate(P.forall("c", Contact, owns.or_(c.has("age").not_()))) is True  # those without an age may have none
assert evaluate(P.count("c", Contact, c.has("age"))) == 1 and evaluate(P.extent(Contact)) == (ann, bob)
assert evaluate(P.exists("c", Contact, c.name.eq("bob").and_(owns.not_()))) is True
assert evaluate(P.choice((0.5, x), (0.5, False)), {"x": None}) is None  # Kleene's disjunction
assert evaluate(P.choice((0.5, x), (0.5, True)), {"x": None}) is True
assert evaluate(P.choice((0.5, False), (0.5, False))) is False
assert evaluate(P.linked(c, "phones", p), {"c": ann, "p": None}) is None  # an unknown end: unknown
listed(store, store.Contact().name("cy").create())
assert len(evaluate(P.extent(Contact))) == 2 and len(P.Evaluator(store)(P.extent(Contact))) == 3  # read once each
with raises(TypeError, match="linked expects an object as its source, got int"):
    evaluate(P.linked(c, "phones", p), {"c": 1, "p": ann})
with raises(TypeError, match="linked expects an object as its target, got str"):
    evaluate(P.linked(c, "phones", p), {"c": ann, "p": "x"})
with raises(TypeError, match="'Contact' has no adjacency 'pagers'"):
    evaluate(P.linked(c, "pagers", p), {"c": ann, "p": ann})
with raises(AttributeError, match="no schema registered as 'Nope'"):
    evaluate(P.extent("Nope"))
with raises(TypeError, match="a choice's options must be bools, got int"):
    evaluate(P.choice((1.0, 3)))
# A relation of more than two links: linked names the link.
Assigned = S.OfRelation.Builder().name("Assigned").links("desk", "person", "seat").create()
Person = S.OfObject.Builder().name("Person").ref().relations(lambda r: r.name("desks").of(Assigned).me("person")).create()
Seat = S.OfObject.Builder().name("Seat").ref().relations(lambda r: r.name("desks").of(Assigned).me("seat")).create()
Desk = S.OfObject.Builder().name("Desk").ref().singleton("office.Desk").relations(
    lambda r: r.name("assigned").of(Assigned).me("desk")).create()
office = Proxies.OfStore()
for schema in (Assigned, Person, Seat, Desk):
    office.register(schema)
desk, pam, seat = office.singleton("office.Desk"), office.Person().create(), office.Seat().create()
office.Desk(desk).assigned(lambda e: e.person(pam).seat(seat)).update()
at = P.Evaluator(office)
assert at(P.linked(d, "assigned", x, "person"), {"d": desk, "x": pam}) is True
assert at(P.linked(d, "assigned", x, "seat"), {"d": desk, "x": seat}) is True
assert at(P.linked(d, "assigned", x, "seat"), {"d": desk, "x": pam}) is False
with raises(ValueError, match="linked needs a link for 'assigned', whose relation has links 'person', 'seat'"):
    at(P.linked(d, "assigned", x), {"d": desk, "x": pam})
seated = C.OfPredicate.Builder().name("Seated").symbols({"d": Desk, "x": Person}).rule(P.linked(d, "assigned", x)).create()
assert Q.Scan(office).explain(seated) == ["d: scan Desk", "x: scan Person, then 1 test"]  # which link: no hop
named = C.OfPredicate.Builder(seated).rule(P.linked(d, "assigned", x, "person")).clone()
assert Q.Scan(office).explain(named) == ["d: scan Desk", "x: d.assigned to person, any number, then 1 test"]
assert list(Q.Scan(office).select(named)) == [{"d": desk, "x": pam}]

## ALG-03 · Predicates written in the algebra: about the whole store, through links, and as data

In [ ]:
store = book()
ann = store.Contact().name("ann").age(30).phones(lambda e: e.phone(lambda q: q.number("1"))).create()
bob = store.Contact().name("bob").create()
listed(store, ann, bob)
EveryoneHasAPhone = C.OfPredicate.Builder().name("EveryoneHasAPhone").rule(mandatory).create()  # no symbols
assert V.Validate(store, [EveryoneHasAPhone])(Contact, ann) == ["the store: 'EveryoneHasAPhone' does not hold"]
query = Q.Scan(store)
assert list(query.select(EveryoneHasAPhone)) == [] and query.explain(EveryoneHasAPhone) == ["first 1 test"]
store.Contact(bob).phones(lambda e: e.phone(lambda q: q.number("2"))).update()
assert list(query.select(EveryoneHasAPhone)) == [{}]  # one match, empty, when it holds
Numbered = C.OfPredicate.Builder().name("Numbered").symbols({"c": Contact, "p": Phone}).rule(
    P.linked(c, "phones", p).and_(p.has("number"))).create()
assert query.explain(Numbered) == ["c: scan Contact", "p: c.phones to phone, any number, then 2 tests"]
assert [(m["c"].name, m["p"].number) for m in query.select(Numbered)] == [("ann", "1"), ("bob", "2")]
Owner = C.OfPredicate.Builder().name("Owner").symbols({"c": Contact, "p": Phone}).rule(P.linked(p, "owners", c)).create()
assert query.explain(Owner) == ["p: scan Phone", "c: p.owners to owner, at most one, then 1 test"]
assert query.explain(C.OfPredicate.Builder(Owner).rule(P.linked(p, "owners", c, "phone")).clone()) == [
    "c: scan Contact", "p: scan Phone, then 1 test"]  # its own link: not a hop
assert query.explain(C.OfPredicate.Builder(Owner).rule(P.linked(c, "nope", p)).clone()) == [
    "c: scan Contact", "p: scan Phone, then 1 test"]  # no such adjacency
assert query.explain(C.OfPredicate.Builder(Owner).rule(P.linked(c, "phones", c)).clone()) == [
    "c: scan Contact, then 1 test", "p: scan Phone"]  # back to itself
predicates = C.OfSet.Builder().predicates(EveryoneHasAPhone, Numbered, lambda b: b.name("Possible").rule(possible)).create()
schemas = C.OfStore(store)
text = JSON.ToJSON(schemas).Reachable(C.OfSet.Schema, predicates)
assert '"$schema": "Patterns.OfForall"' in text and '"$schema": "Patterns.OfOption"' in text
copy = JSON.FromJSON(schemas).Reachable(C.OfSet.Schema, text)
assert all(Terms.same(a.rule, b.rule) for a, b in zip(copy.predicates, predicates.predicates))
assert copy.predicates[2].rule.body.options[0].weight == 0.35